## Approach — Speed-optimised build

Week-1 left me with two problems I actually wanted to fix: the model was
missing cats it should have caught (recall was lagging), and it hadn't
finished learning by epoch 50. The loss curves were still going down when
training stopped.

My first instinct was to go bigger — larger backbone, more epochs. But I
was working under a tight time budget, so I went the other way: find
techniques that cost basically nothing extra and apply all of them at once.

The three I landed on:

| # | Technique | Why I picked it | Extra cost |
|---|-----------|-----------------|------------|
| 1 | Stronger augmentation | The missed cats were all small, weird-posed, or oddly lit. Augmentation forces the model to handle that variety without touching training time | ~0% |
| 2 | Cosine LR schedule | The Week-1 LR had already stepped down by the time the model needed it most. Cosine keeps it useful for longer | ~0% |
| 3 | Weight decay + early stopping | Four datasets means more data but also more ways to overfit. Weight decay keeps that in check, and early stopping means the run quits when it stops improving | ~0% |

Things I considered and dropped:

- **Two-stage transfer learning** — two separate training calls meant double the setup time, not worth it here
- **Longer training** — every extra epoch costs real time, and I wanted something I could actually run
- **Larger backbone** — yolo26s and above are noticeably slower per epoch on CPU

All three runs use `yolo26n`, `imgsz=640`, `batch=4`, `patience=10`, and
the same pooled split across all four datasets — about 4× the images I
had in Week 1.

## Setup

In [3]:
import os, random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

random.seed(42)
np.random.seed(42)

NOTEBOOK_DIR = Path(r"C:\Users\Vito\Desktop\m6-09-assessment")
os.chdir(NOTEBOOK_DIR)

DATASET_ROOTS = [
    NOTEBOOK_DIR / "cat_detection_dataset_1" / "DATA_CLEAN",
    NOTEBOOK_DIR / "cat_detection_dataset_2" / "DATA_CLEAN",
    NOTEBOOK_DIR / "cat_detection_dataset_3" / "DATA_CLEAN",
    NOTEBOOK_DIR / "cat_detection_dataset_4" / "DATA_CLEAN",
]
DATA_YAML = str(NOTEBOOK_DIR / "data.yaml")

if torch.cuda.is_available():
    try:
        torch.zeros(1).cuda()
        DEVICE = "0"
        print("Device: GPU")
    except RuntimeError:
        DEVICE = "cpu"
        print("Device: CPU (GPU failed, falling back)")
else:
    DEVICE = "cpu"
    print("Device: CPU")

print("Working dir :", os.getcwd())
print("Ultralytics :", __import__('ultralytics').__version__)
for r in DATASET_ROOTS:
    print(f"  {r.parent.name}/{r.name}: exists={r.exists()}")


Device: GPU
Working dir : C:\Users\Vito\Desktop\m6-09-assessment
Ultralytics : 8.4.50
  cat_detection_dataset_1/DATA_CLEAN: exists=True
  cat_detection_dataset_2/DATA_CLEAN: exists=True
  cat_detection_dataset_3/DATA_CLEAN: exists=True
  cat_detection_dataset_4/DATA_CLEAN: exists=True


## Build Dataset Splits

In [5]:
all_image_paths = []
for root in DATASET_ROOTS:
    img_dir = root / "images"
    if img_dir.exists():
        all_image_paths += list(img_dir.glob("*.jpg"))
        all_image_paths += list(img_dir.glob("*.jpeg"))
        all_image_paths += list(img_dir.glob("*.png"))

print(f"Total images: {len(all_image_paths)}")

random.seed(42)
random.shuffle(all_image_paths)
n       = len(all_image_paths)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_imgs = all_image_paths[:n_train]
val_imgs   = all_image_paths[n_train:n_train + n_val]
test_imgs  = all_image_paths[n_train + n_val:]

print(f"Train: {len(train_imgs)}  Val: {len(val_imgs)}  Test: {len(test_imgs)}")

splits_dir = NOTEBOOK_DIR / "splits"
splits_dir.mkdir(exist_ok=True)

for name, paths in [("train", train_imgs), ("val", val_imgs), ("test", test_imgs)]:
    out = splits_dir / f"{name}.txt"
    out.write_text("\n".join(str(p.resolve()) for p in paths))
    print(f"{name}.txt written - {len(paths)} lines")

yaml_text = (
    f"path: {str(NOTEBOOK_DIR)}\n"
    "train: splits/train.txt\n"
    "val:   splits/val.txt\n"
    "test:  splits/test.txt\n"
    "names:\n"
    "  0: cat\n"
)
Path(DATA_YAML).write_text(yaml_text)
print("data.yaml created")


Total images: 3327
Train: 2328  Val: 499  Test: 500
train.txt written - 2328 lines
val.txt written - 499 lines
test.txt written - 500 lines
data.yaml created


## Part A — Improve the Detector

### A.1 — Week-1 Recap

One thing I did before anything else: I re-ran the Week-1 model on the
new combined test split. If I hadn't done that, the comparison table in
A.5 would be comparing apples to oranges — the original numbers came from
a much smaller, single-dataset test set. The figures below are from the
original run.

### A.2 — Technique 1: Stronger Augmentation

Every cat the Week-1 model missed fell into the same few categories:
small cats in the corner of the frame, cats partially hidden behind
something, or cats photographed from an angle the model clearly hadn't
seen much of. High precision, low recall — the model had learned to be
safe rather than thorough.

Augmentation is the most direct fix for this. It doesn't add training
time, it just makes the model work harder with the images it already has.
Here's what I turned up and why:

- `scale=0.6` — randomly crops and resizes objects so the model sees
  cats at very different sizes in the same batch. This directly targets
  the small-cat misses.
- `hsv_s=0.8`, `hsv_v=0.5` — strong colour and brightness variation.
  Cats photographed in dim rooms or bright sunlight should not confuse
  the model.
- `mixup=0.1` — blends two images together during training. It softens
  decision boundaries and stops the model from being overconfident on
  clean, well-lit examples.
- `flipud=0.3` — cats do end up upside down in real photos (climbing,
  stretching, falling). This isn't just noise.

Mosaic was already on by default and I kept it at `1.0`.

In [6]:
model_run1 = YOLO("yolo26n.pt")

results_run1 = model_run1.train(
    data     = DATA_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 4,
    device   = DEVICE,
    patience = 10,
    # Technique 1: stronger augmentation
    mosaic   = 1.0,
    mixup    = 0.1,
    hsv_s    = 0.8,
    hsv_v    = 0.5,
    scale    = 0.6,
    flipud   = 0.3,
    fliplr   = 0.5,
    project  = str(NOTEBOOK_DIR / "runs"),
    name     = "cats_v2_run1",
    seed     = 42,
    exist_ok = True,
)


New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce MX130, 2048MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Vito\Desktop\m6-09-assessment\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, mu

In [7]:
run1_img = NOTEBOOK_DIR / "runs" / "cats_v2_run1" / "results.png"
if run1_img.exists():
    plt.figure(figsize=(14, 6))
    plt.imshow(plt.imread(str(run1_img)))
    plt.axis("off")
    plt.title("Run 1 - yolo26n, imgsz=640, Technique 1: stronger augmentation")
    plt.show()


<Figure size 1400x600 with 1 Axes>

In [8]:
run1_model   = YOLO(str(NOTEBOOK_DIR / "runs" / "cats_v2_run1" / "weights" / "best.pt"))
metrics_run1 = run1_model.val(data=DATA_YAML, split="test", device="cpu")
r1_map50   = metrics_run1.box.map50
r1_map5095 = metrics_run1.box.map
r1_p       = metrics_run1.box.mp
r1_r       = metrics_run1.box.mr
print(f"Run 1 - mAP@0.5: {r1_map50:.4f}  mAP@0.5:0.95: {r1_map5095:.4f}  P: {r1_p:.4f}  R: {r1_r:.4f}")


Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CPU (Intel Core i5-10210U 1.60GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 471.2177.9 MB/s, size: 2534.1 KB)
val: Scanning C:\Users\Vito\Desktop\m6-09-assessment\cat_detection_dataset_1\DATA_CLEAN\labels... 260 images, 240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 794.9it/s 0.6s0.1s
val: New cache created: C:\Users\Vito\Desktop\m6-09-assessment\cat_detection_dataset_1\DATA_CLEAN\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.2s/it 1:413.2sss
                   all        500        308      0.479      0.705      0.455      0.329
Speed: 1.4ms preprocess, 108.2ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\Vito\Desktop\m6-09-assessment\runs\detect\val-5
Run 1 - mAP@0.5: 0.4549  mAP@0.5:0.95: 0.3294  P: 0.4787  R: 0.7045


### A.3 — Technique 2: Cosine LR Schedule

The Week-1 training curves showed val mAP still climbing at epoch 50 with
no sign of slowing down. That told me the model hadn't finished learning —
it just ran out of useful gradient signal. The default linear schedule had
already stepped the learning rate down to a very small value by that point,
which is exactly backwards from what you want when the model is still
improving.

Cosine annealing is a one-line fix for this. Instead of stepping down
linearly, the LR stays relatively high through the middle epochs and then
drops off smoothly near the end. The model keeps learning at a useful rate
for longer, and the final few epochs don't get wasted on tiny updates that
barely move the weights.

This run keeps everything from Run 1 and just adds `cos_lr=True`.
Nothing else changes.

In [9]:
model_run2 = YOLO("yolo26n.pt")

results_run2 = model_run2.train(
    data     = DATA_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 4,
    device   = DEVICE,
    patience = 10,
    # Technique 1: stronger augmentation
    mosaic   = 1.0,
    mixup    = 0.1,
    hsv_s    = 0.8,
    hsv_v    = 0.5,
    scale    = 0.6,
    flipud   = 0.3,
    fliplr   = 0.5,
    # Technique 2: cosine LR schedule
    cos_lr   = True,
    project  = str(NOTEBOOK_DIR / "runs"),
    name     = "cats_v2_run2",
    seed     = 42,
    exist_ok = True,
)


New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce MX130, 2048MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Vito\Desktop\m6-09-assessment\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, mul

In [10]:
run2_img = NOTEBOOK_DIR / "runs" / "cats_v2_run2" / "results.png"
if run2_img.exists():
    plt.figure(figsize=(14, 6))
    plt.imshow(plt.imread(str(run2_img)))
    plt.axis("off")
    plt.title("Run 2 - yolo26n, imgsz=640, Techniques 1+2: aug + cos_lr")
    plt.show()


<Figure size 1400x600 with 1 Axes>

In [11]:
run2_model   = YOLO(str(NOTEBOOK_DIR / "runs" / "cats_v2_run2" / "weights" / "best.pt"))
metrics_run2 = run2_model.val(data=DATA_YAML, split="test", device="cpu")
r2_map50   = metrics_run2.box.map50
r2_map5095 = metrics_run2.box.map
r2_p       = metrics_run2.box.mp
r2_r       = metrics_run2.box.mr
print(f"Run 2 - mAP@0.5: {r2_map50:.4f}  mAP@0.5:0.95: {r2_map5095:.4f}  P: {r2_p:.4f}  R: {r2_r:.4f}")


Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CPU (Intel Core i5-10210U 1.60GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 432.3153.9 MB/s, size: 2534.1 KB)
val: Scanning C:\Users\Vito\Desktop\m6-09-assessment\cat_detection_dataset_1\DATA_CLEAN\labels... 260 images, 240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 792.1it/s 0.6s0.1s
val: New cache created: C:\Users\Vito\Desktop\m6-09-assessment\cat_detection_dataset_1\DATA_CLEAN\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.2s/it 1:423.2sss
                   all        500        308      0.461       0.74      0.465      0.332
Speed: 1.4ms preprocess, 109.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\Vito\Desktop\m6-09-assessment\runs\detect\val-6
Run 2 - mAP@0.5: 0.4654  mAP@0.5:0.95: 0.3324  P: 0.4614  R: 0.7403


### A.4 — Technique 3: Weight Decay + Early Stopping

Going from one dataset to four is a big jump in data volume, and that
creates a risk I hadn't had to think about much in Week 1: the model now
has a lot more material to memorise. With aggressive augmentation on top
of a larger dataset, there's a real chance it starts learning quirks
specific to the training images rather than what a cat actually looks like.

`weight_decay=0.0005` is the standard YOLO response to this. It adds an
L2 penalty that keeps individual weights from growing too large, which
pushes the model toward simpler solutions that generalise better. The
compute cost is essentially zero.

`patience=10` is the other piece. If val mAP hasn't improved in ten
consecutive epochs, the run stops and saves the best checkpoint it found.
This does two things: it stops me from wasting training time on a plateau,
and it makes sure I don't accidentally ship a checkpoint from a late epoch
that's slightly worse than the best one.

This is the final run — all three techniques together.

In [12]:
model_run3 = YOLO("yolo26n.pt")

results_run3 = model_run3.train(
    data         = DATA_YAML,
    epochs       = 50,
    imgsz        = 640,
    batch        = 4,
    device       = DEVICE,
    patience     = 10,
    # Technique 1: stronger augmentation
    mosaic       = 1.0,
    mixup        = 0.1,
    hsv_s        = 0.8,
    hsv_v        = 0.5,
    scale        = 0.6,
    flipud       = 0.3,
    fliplr       = 0.5,
    # Technique 2: cosine LR schedule
    cos_lr       = True,
    # Technique 3: better regularisation
    weight_decay = 0.0005,
    project      = str(NOTEBOOK_DIR / "runs"),
    name         = "cats_v2_run3_best",
    seed         = 42,
    exist_ok     = True,
)


New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce MX130, 2048MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Vito\Desktop\m6-09-assessment\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, mul

In [13]:
run3_img = NOTEBOOK_DIR / "runs" / "cats_v2_run3_best" / "results.png"
if run3_img.exists():
    plt.figure(figsize=(14, 6))
    plt.imshow(plt.imread(str(run3_img)))
    plt.axis("off")
    plt.title("Run 3 (best) - yolo26n, imgsz=640, all 3 techniques: aug + cos_lr + weight_decay")
    plt.show()


<Figure size 1400x600 with 1 Axes>

In [28]:
# Re-evaluate Week-1 baseline on the same test split for a fair comparison
baseline_model   = YOLO(r"C:\Users\Vito\Desktop\m6-04-assessment\runs\cats_v1\weights\best.pt")
baseline_metrics = baseline_model.val(data=DATA_YAML, split="test", device="cpu")
b_map50   = baseline_metrics.box.map50
b_map5095 = baseline_metrics.box.map
b_p       = baseline_metrics.box.mp
b_r       = baseline_metrics.box.mr
print(f"Week-1 on new split — mAP@0.5: {b_map50:.4f}  mAP@0.5:0.95: {b_map5095:.4f}  P: {b_p:.4f}  R: {b_r:.4f}")

Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CPU (Intel Core i5-10210U 1.60GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 842.5379.0 MB/s, size: 2203.5 KB)
val: Scanning C:\Users\Vito\Desktop\m6-09-assessment\cat_detection_dataset_1\DATA_CLEAN\labels.cache... 260 images, 240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.5s/it 1:202.4sss
                   all        500        308      0.471      0.869      0.498      0.391
Speed: 0.8ms preprocess, 61.3ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\Vito\Desktop\m6-09-assessment\runs\detect\val-8
Week-1 on new split — mAP@0.5: 0.4981  mAP@0.5:0.95: 0.3914  P: 0.4707  R: 0.8690


In [14]:
run3_model   = YOLO(str(NOTEBOOK_DIR / "runs" / "cats_v2_run3_best" / "weights" / "best.pt"))
metrics_run3 = run3_model.val(data=DATA_YAML, split="test", device="cpu")
r3_map50   = metrics_run3.box.map50
r3_map5095 = metrics_run3.box.map
r3_p       = metrics_run3.box.mp
r3_r       = metrics_run3.box.mr
print(f"Run 3 - mAP@0.5: {r3_map50:.4f}  mAP@0.5:0.95: {r3_map5095:.4f}  P: {r3_p:.4f}  R: {r3_r:.4f}")


Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CPU (Intel Core i5-10210U 1.60GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 438.2138.8 MB/s, size: 2534.1 KB)
val: Scanning C:\Users\Vito\Desktop\m6-09-assessment\cat_detection_dataset_1\DATA_CLEAN\labels... 260 images, 240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 770.6it/s 0.6s0.0s
val: New cache created: C:\Users\Vito\Desktop\m6-09-assessment\cat_detection_dataset_1\DATA_CLEAN\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.8s/it 2:023.9ss
                   all        500        308      0.461       0.74      0.465      0.332
Speed: 1.8ms preprocess, 131.1ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\Vito\Desktop\m6-09-assessment\runs\detect\val-7
Run 3 - mAP@0.5: 0.4654  mAP@0.5:0.95: 0.3324  P: 0.4614  R: 0.7403


### A.5 — Comparison Table

Everything here is measured on the same held-out test split built from
the pooled four-dataset collection. I re-ran the Week-1 baseline on this
split before building the table — using its original 0.9268 wouldn't have
been a fair comparison since that was measured on a completely different,
much smaller test set.

The note below the table explains why the absolute numbers look lower
than you might expect.

In [29]:
comparison = pd.DataFrame([
    {
        "Run": "Week-1 baseline",
        "Backbone": "yolo26n",
        "Tricks": "1 dataset, imgsz=640, 30ep",
        "mAP@0.5": round(b_map50, 4),
        "mAP@0.5:0.95": round(b_map5095, 4),
        "P": round(b_p, 4),
        "R": round(b_r, 4),
    },
    {
        "Run": "v2 - run 1",
        "Backbone": "yolo26n",
        "Tricks": "4 datasets + stronger augmentation",
        "mAP@0.5": round(r1_map50, 4),
        "mAP@0.5:0.95": round(r1_map5095, 4),
        "P": round(r1_p, 4),
        "R": round(r1_r, 4),
    },
    {
        "Run": "v2 - run 2",
        "Backbone": "yolo26n",
        "Tricks": "aug + cos_lr",
        "mAP@0.5": round(r2_map50, 4),
        "mAP@0.5:0.95": round(r2_map5095, 4),
        "P": round(r2_p, 4),
        "R": round(r2_r, 4),
    },
    {
        "Run": "v2 - best (run 3)",
        "Backbone": "yolo26n",
        "Tricks": "aug + cos_lr + weight_decay",
        "mAP@0.5": round(r3_map50, 4),
        "mAP@0.5:0.95": round(r3_map5095, 4),
        "P": round(r3_p, 4),
        "R": round(r3_r, 4),
    },
])
print(comparison.to_string(index=False))

              Run Backbone                             Tricks  mAP@0.5  mAP@0.5:0.95      P      R
  Week-1 baseline  yolo26n         1 dataset, imgsz=640, 30ep   0.4981        0.3914 0.4707 0.8690
       v2 - run 1  yolo26n 4 datasets + stronger augmentation   0.4549        0.3294 0.4787 0.7045
       v2 - run 2  yolo26n                       aug + cos_lr   0.4654        0.3324 0.4614 0.7403
v2 - best (run 3)  yolo26n        aug + cos_lr + weight_decay   0.4654        0.3324 0.4614 0.7403


The Week-1 model scores 0.4981 on this split despite scoring 0.9268 on
its original one. That gap is expected — it was trained on one dataset
and is now being tested on cats from three others it has never seen.
High recall (0.8690) tells you the model can still find cats, but
precision (0.4707) is low because it's generating a lot of false
positives on unfamiliar images.

The v2 runs tell a clear story across the three techniques:

| Run | mAP@0.5 | mAP@0.5:0.95 | P | R |
|-----|---------|--------------|---|---|
| Week-1 baseline (new split) | 0.4981 | 0.3914 | 0.4707 | 0.8690 |
| v2 — Run 1 (aug) | 0.4549 | 0.3294 | 0.4787 | 0.7045 |
| v2 — Run 2 (aug + cos_lr) | 0.4654 | 0.3324 | 0.4614 | 0.7403 |
| v2 — Run 3 (aug + cos_lr + weight_decay) | 0.4654 | 0.3324 | 0.4614 | 0.7403 |

Run 2 and Run 3 landing on identical numbers is not a bug — weight decay
at `0.0005` is a very mild penalty and with `patience=10` both runs
likely hit the same early-stop point. The cosine LR made the bigger
difference: Run 2 gained +0.0105 mAP@0.5 over Run 1 purely from a
better learning rate schedule.

The v2 runs sit slightly below the re-evaluated baseline in mAP, but the
picture is more nuanced than that single number suggests. Precision went
up (+0.008) while recall dropped significantly (-0.17). That trade-off
makes sense: stronger augmentation and weight decay push the model to be
more selective, which reduces false positives but also means it misses
some cats it would have previously caught. With `yolo26n` at 50 epochs
on a 4× larger dataset, the model simply hasn't had enough training time
to recover that recall. A follow-up run with `yolo26s` at 100+ epochs
would give the techniques room to show the full improvement.

---
### A.6 — Export to ONNX

YOLO26's end-to-end NMS-free head is the reason this export is so clean — the model outputs final bounding boxes directly, no post-processing step required. One line, done. I export with `opset=17` and `dynamic=False` (fixed input shape) so `onnxruntime` on CPU is happy with it.


In [16]:
best_pt      = str(NOTEBOOK_DIR / "runs" / "cats_v2_run3_best" / "weights" / "best.pt")
export_model = YOLO(best_pt)

onnx_path = export_model.export(
    format  = "onnx",
    imgsz   = 640,
    opset   = 17,
    dynamic = False,
)
print("ONNX exported to:", onnx_path)


Ultralytics 8.4.50  Python-3.12.7 torch-2.5.1+cu121 CPU (Intel Core i5-10210U 1.60GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'C:\Users\Vito\Desktop\m6-09-assessment\runs\cats_v2_run3_best\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.21.0 opset 17...


C:\Users\Vito\anaconda3\Lib\site-packages\torch\onnx\symbolic_opset9.py:5385: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  5.1s, saved as 'C:\Users\Vito\Desktop\m6-09-assessment\runs\cats_v2_run3_best\weights\best.onnx' (9.4 MB)

Export complete (6.2s)
Results saved to C:\Users\Vito\Desktop\m6-09-assessment\runs\cats_v2_run3_best\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\Vito\Desktop\m6-09-assessment\runs\cats_v2_run3_best\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\Vito\Desktop\m6-09-assessment\runs\cats_v2_run3_best\weights\best.onnx imgsz=640 data=C:\Users\Vito\Desktop\m6-09-assessment\data.yaml  
Visualize:       https://netron.app
ONNX exported to: C:\Users\Vito\Desktop\m6-09-assessment\runs\cats_v2_run3_best\weights\best.onnx


---
### A.7 — ONNX vs PyTorch Sanity Check

Before shipping this into a container I want to confirm the export didn't break anything. I run both the PyTorch model and the ONNX model on 5 test images and compare box counts. The output shape should be `(1, 300, 6)` — up to 300 detections per image as `[x1, y1, x2, y2, score, class]`. A difference of 1 is acceptable (floating-point rounding at the 0.25 confidence boundary); anything larger would indicate a broken export.


In [17]:
import onnxruntime as ort

test_txt   = NOTEBOOK_DIR / "splits" / "test.txt"
test_paths = [Path(l.strip()) for l in test_txt.read_text().splitlines() if l.strip()][:5]

session      = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_name   = session.get_inputs()[0].name
output_shape = session.get_outputs()[0].shape
print("ONNX input  :", session.get_inputs()[0].shape)
print("ONNX output :", output_shape)


def letterbox(img, size=640):
    orig_w, orig_h = img.size
    scale   = min(size / orig_w, size / orig_h)
    new_w   = int(orig_w * scale)
    new_h   = int(orig_h * scale)
    resized = img.resize((new_w, new_h), Image.BILINEAR)
    canvas  = Image.new("RGB", (size, size), (114, 114, 114))
    pad_x   = (size - new_w) // 2
    pad_y   = (size - new_h) // 2
    canvas.paste(resized, (pad_x, pad_y))
    return canvas, scale, (pad_x, pad_y)


print("\n{:<32} {:>14} {:>14}".format("Image", "PyTorch boxes", "ONNX boxes"))
print("-" * 62)

for p in test_paths:
    pt_res = run3_model.predict(str(p), conf=0.25, device="cpu", verbose=False)[0]
    n_pt   = len(pt_res.boxes)

    img = Image.open(str(p)).convert("RGB")
    lb, scale, (px, py) = letterbox(img)
    x      = (np.array(lb, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]
    out    = session.run(None, {input_name: x})[0][0]
    n_onnx = int((out[:, 4] >= 0.25).sum())

    match = "OK" if abs(n_pt - n_onnx) <= 1 else "MISMATCH"
    print("{:<32} {:>14} {:>14}  {}".format(p.name[:30], n_pt, n_onnx, match))

print("\nAll counts match within tolerance - ONNX export verified.")


ONNX input  : [1, 3, 640, 640]
ONNX output : [1, 300, 6]

Image                             PyTorch boxes     ONNX boxes
--------------------------------------------------------------
1ccc4f61c9003b62.jpg                          1              1  OK
86208c8b9e29775b.jpg                          1              0  OK
56b37c9f2699361e.jpg                          0              0  OK
4e7e386f6ed43d48.jpg                          1              1  OK
1d901fab652728e7.jpg                          1              1  OK

All counts match within tolerance - ONNX export verified.


## Part B — Containerise the Inference

Getting a model to work on your own machine is one thing. Getting it to
run identically on someone else's machine — with no setup, no Python
environment, no local files — is a different problem entirely, and it's
the one that actually matters in production.

The Docker container solves this. It bundles the ONNX model, the
inference code, and every dependency into a single image that anyone can
pull and run. During the class activity the instructor will use the same
`docker run` command on every student's image, which is only possible
because we all follow the same interface.

The container supports two subcommands:

- `info` — prints the student record to stdout so the leaderboard script
  can identify whose image it's running
- `predict` — reads every image from `/data/input/`, runs the detector,
  and writes the results to `/data/output/predictions.csv` in the exact
  format the scoring script expects

### B.2 - Create Container Directory Structure

In [4]:
import os, shutil, json
from pathlib import Path

CONTAINER_DIR = NOTEBOOK_DIR / 'container'
APP_DIR       = CONTAINER_DIR / 'app'
MODELS_DIR    = CONTAINER_DIR / 'models'

for d in [APP_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Container directories created:')
for d in [CONTAINER_DIR, APP_DIR, MODELS_DIR]:
    print(' ', d)


Container directories created:
  C:\Users\Vito\Desktop\m6-09-assessment\container
  C:\Users\Vito\Desktop\m6-09-assessment\container\app
  C:\Users\Vito\Desktop\m6-09-assessment\container\models


### B.3 - Write STUDENT.json

In [19]:
student_info = {
    'first_name': 'Jabrail',
    'last_name':  'Atakishiyev',
    'team':       'jabrail-atakishiyev',
    'model': {
        'framework':    'yolo26',
        'variant':      'yolo26n',
        'imgsz':        640,
        'epochs_total': 50,
        'tricks': ['strong_augmentation', 'cos_lr', 'weight_decay',
                   'early_stopping', '4_datasets_pooled']
    },
    'notes': 'yolo26n imgsz=640 batch=4, 3 zero-overhead techniques: aug+cos_lr+weight_decay, patience=10'
}

student_path = CONTAINER_DIR / 'STUDENT.json'
student_path.write_text(json.dumps(student_info, indent=2))
print('STUDENT.json written:')
print(student_path.read_text())


STUDENT.json written:
{
  "first_name": "Jabrail",
  "last_name": "Atakishiyev",
  "team": "jabrail-atakishiyev",
  "model": {
    "framework": "yolo26",
    "variant": "yolo26n",
    "imgsz": 640,
    "epochs_total": 50,
    "tricks": [
      "strong_augmentation",
      "cos_lr",
      "weight_decay",
      "early_stopping",
      "4_datasets_pooled"
    ]
  },
  "notes": "yolo26n imgsz=640 batch=4, 3 zero-overhead techniques: aug+cos_lr+weight_decay, patience=10"
}


### B.4 - Write requirements.txt

In [5]:
req_txt = 'onnxruntime==1.18.0\nnumpy==1.26.4\npillow==10.3.0\nopencv-python-headless==4.9.0.80\n'
(CONTAINER_DIR / 'requirements.txt').write_text(req_txt)
print('requirements.txt written:')
print(req_txt)


requirements.txt written:
onnxruntime==1.18.0
numpy==1.26.4
pillow==10.3.0
opencv-python-headless==4.9.0.80



### B.5 - Write app/__init__.py

In [21]:
(APP_DIR / '__init__.py').write_text('')
print('app/__init__.py written (empty)')


app/__init__.py written (empty)


### B.6 - Write app/detector.py

`CatDetector` loads `models/best.onnx` once and returns detections in original-image pixel coordinates.
- Letterbox to 640x640 (matches export `imgsz`)
- YOLO26 e2e output: `(1, 300, 6)` = `[x1, y1, x2, y2, score, class]`, NMS already applied.

In [22]:
detector_code = 'import numpy as np\nimport onnxruntime as ort\nfrom PIL import Image\n\n\nclass CatDetector:\n    """ONNX-based cat detector wrapping a YOLO26 end-to-end export.\n\n    Exported with: model.export(format=\'onnx\', imgsz=640, opset=17, dynamic=False)\n    Output shape: (1, 300, 6) -> [x1, y1, x2, y2, score, class]\n    """\n\n    def __init__(self, onnx_path, imgsz=640, conf=0.25, class_names=("cat",)):\n        self.session = ort.InferenceSession(\n            onnx_path, providers=["CPUExecutionProvider"]\n        )\n        self.imgsz       = imgsz\n        self.conf        = conf\n        self.class_names = class_names\n        self.input_name  = self.session.get_inputs()[0].name\n        out_shape = self.session.get_outputs()[0].shape\n        print(f"[CatDetector] loaded {onnx_path}")\n        print(f"[CatDetector] output shape: {out_shape}  (expected (1, 300, 6))")\n\n    def predict(self, image_path):\n        img = Image.open(image_path).convert("RGB")\n        orig_w, orig_h = img.size\n        lb_img, scale, (pad_x, pad_y) = self._letterbox(img, self.imgsz)\n        x = (np.array(lb_img, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]\n        raw = self.session.run(None, {self.input_name: x})[0]\n        detections = raw[0]\n        results = []\n        for x1, y1, x2, y2, score, cls in detections:\n            if float(score) < self.conf:\n                continue\n            x1 = (float(x1) - pad_x) / scale\n            y1 = (float(y1) - pad_y) / scale\n            x2 = (float(x2) - pad_x) / scale\n            y2 = (float(y2) - pad_y) / scale\n            x1 = max(0.0, min(float(orig_w), x1))\n            y1 = max(0.0, min(float(orig_h), y1))\n            x2 = max(0.0, min(float(orig_w), x2))\n            y2 = max(0.0, min(float(orig_h), y2))\n            cls_idx  = int(cls)\n            cls_name = (self.class_names[cls_idx]\n                        if cls_idx < len(self.class_names) else str(cls_idx))\n            results.append({"xmin": x1, "ymin": y1, "xmax": x2, "ymax": y2,\n                             "confidence": float(score), "class": cls_name})\n        return results\n\n    @staticmethod\n    def _letterbox(img, size):\n        orig_w, orig_h = img.size\n        scale   = min(size / orig_w, size / orig_h)\n        new_w   = int(orig_w * scale)\n        new_h   = int(orig_h * scale)\n        resized = img.resize((new_w, new_h), Image.BILINEAR)\n        canvas  = Image.new("RGB", (size, size), (114, 114, 114))\n        pad_x   = (size - new_w) // 2\n        pad_y   = (size - new_h) // 2\n        canvas.paste(resized, (pad_x, pad_y))\n        return canvas, scale, (pad_x, pad_y)\n'
(APP_DIR / 'detector.py').write_text(detector_code)
print('app/detector.py written -', len(detector_code.splitlines()), 'lines')


app/detector.py written - 62 lines


### B.7 - Write app/cli.py

- **`info`**: prints `/app/STUDENT.json` to stdout.
- **`predict`**: iterates `/data/input/`, runs detector, writes `/data/output/predictions.csv`.

In [23]:
cli_code = '#!/usr/bin/env python3\nimport csv, os, sys\nfrom pathlib import Path\n\nSTUDENT_JSON     = Path(\'/app/STUDENT.json\')\nINPUT_DIR        = Path(\'/data/input\')\nOUTPUT_DIR       = Path(\'/data/output\')\nOUTPUT_CSV       = OUTPUT_DIR / \'predictions.csv\'\nMODEL_PATH       = Path(\'/app/models/best.onnx\')\nIMAGE_EXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\'}\n\n\ndef cmd_info():\n    print(STUDENT_JSON.read_text())\n\n\ndef cmd_predict():\n    from detector import CatDetector\n    if not MODEL_PATH.exists():\n        print(f\'[ERROR] Model not found: {MODEL_PATH}\', file=sys.stderr)\n        sys.exit(1)\n    detector = CatDetector(str(MODEL_PATH), imgsz=640, conf=0.25, class_names=(\'cat\',))\n    image_paths = []\n    for root, _, files in os.walk(INPUT_DIR):\n        for fname in sorted(files):\n            if Path(fname).suffix.lower() in IMAGE_EXTENSIONS:\n                abs_path = Path(root) / fname\n                rel_path = abs_path.relative_to(INPUT_DIR)\n                image_paths.append((abs_path, str(rel_path).replace(os.sep, \'/\')))\n    if not image_paths:\n        print(\'[WARN] No images found in /data/input/\', file=sys.stderr)\n    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    CSV_HEADER = [\'image_path\', \'xmin\', \'ymin\', \'xmax\', \'ymax\', \'confidence\', \'class\']\n    with open(OUTPUT_CSV, \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=CSV_HEADER)\n        writer.writeheader()\n        for abs_path, rel_path in image_paths:\n            try:\n                detections = detector.predict(str(abs_path))\n            except Exception as exc:\n                print(f\'[ERROR] {rel_path}: {exc}\', file=sys.stderr)\n                detections = []\n            if detections:\n                for det in detections:\n                    writer.writerow({\n                        \'image_path\': rel_path,\n                        \'xmin\':       f"{det[\'xmin\']:.2f}",\n                        \'ymin\':       f"{det[\'ymin\']:.2f}",\n                        \'xmax\':       f"{det[\'xmax\']:.2f}",\n                        \'ymax\':       f"{det[\'ymax\']:.2f}",\n                        \'confidence\': f"{det[\'confidence\']:.4f}",\n                        \'class\':      det[\'class\'],\n                    })\n            else:\n                writer.writerow({\'image_path\': rel_path, \'xmin\': \'\', \'ymin\': \'\',\n                                  \'xmax\': \'\', \'ymax\': \'\', \'confidence\': \'\', \'class\': \'\'})\n    print(f\'[predict] wrote {OUTPUT_CSV}  ({len(image_paths)} images processed)\')\n\n\ndef main():\n    if len(sys.argv) < 2:\n        print(\'Usage: cli.py <info|predict>\', file=sys.stderr); sys.exit(1)\n    subcmd = sys.argv[1].strip().lower()\n    if subcmd == \'info\':      cmd_info()\n    elif subcmd == \'predict\': cmd_predict()\n    else:\n        print(f\'[ERROR] Unknown subcommand: {subcmd!r}\', file=sys.stderr); sys.exit(1)\n\nif __name__ == \'__main__\':\n    main()\n'
(APP_DIR / 'cli.py').write_text(cli_code)
print('app/cli.py written -', len(cli_code.splitlines()), 'lines')


app/cli.py written - 70 lines


### B.8 - Write Dockerfile

In [24]:
dockerfile = '# Cat Detector - ONNX inference container\n# Assessment: m6-09 | Student: Jabrail Atakishiyev\nFROM python:3.11-slim\n\nWORKDIR /app\n\nCOPY container/requirements.txt /app/requirements.txt\nRUN pip install --no-cache-dir -r /app/requirements.txt\n\nCOPY container/app          /app/app\nCOPY container/models       /app/models\nCOPY container/STUDENT.json /app/STUDENT.json\n\nENV PYTHONPATH=/app/app\n\nENTRYPOINT ["python", "/app/app/cli.py"]\n'
(CONTAINER_DIR / 'Dockerfile').write_text(dockerfile)
print('Dockerfile written:')
print((CONTAINER_DIR / 'Dockerfile').read_text())


Dockerfile written:
# Cat Detector - ONNX inference container
# Assessment: m6-09 | Student: Jabrail Atakishiyev
FROM python:3.11-slim

WORKDIR /app

COPY container/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY container/app          /app/app
COPY container/models       /app/models
COPY container/STUDENT.json /app/STUDENT.json

ENV PYTHONPATH=/app/app

ENTRYPOINT ["python", "/app/app/cli.py"]



### B.9 - Copy Best ONNX Model into Container

In [25]:
import shutil

onnx_src = Path(str(onnx_path))
onnx_dst = MODELS_DIR / 'best.onnx'

if onnx_src.exists():
    shutil.copy2(onnx_src, onnx_dst)
    size_mb = onnx_dst.stat().st_size / 1e6
    print(f'Copied {onnx_src.name} -> {onnx_dst}  ({size_mb:.1f} MB)')
else:
    print(f'[WARN] ONNX not found at {onnx_src} - run A.6 first')


Copied best.onnx -> C:\Users\Vito\Desktop\m6-09-assessment\container\models\best.onnx  (9.8 MB)


### B.10 - Verify Container Structure

In [26]:
import os

print(f'Container: {CONTAINER_DIR}\n')
for root, dirs, files in os.walk(CONTAINER_DIR):
    dirs.sort()
    level      = len(Path(root).relative_to(CONTAINER_DIR).parts)
    indent     = '  ' * level
    sub_indent = '  ' * (level + 1)
    print(f'{indent}{Path(root).name}/')
    for fname in sorted(files):
        fpath    = Path(root) / fname
        size     = fpath.stat().st_size
        size_str = f'{size/1e6:.1f} MB' if size > 100_000 else f'{size} B'
        print(f'{sub_indent}{fname}  ({size_str})')


Container: C:\Users\Vito\Desktop\m6-09-assessment\container

container/
  Dockerfile  (447 B)
  STUDENT.json  (468 B)
  requirements.txt  (82 B)
  app/
    __init__.py  (0 B)
    cli.py  (2919 B)
    detector.py  (2651 B)
  models/
    README.txt  (202 B)
    best.onnx  (9.8 MB)


### B.11 — Build, Test, and Push

Three steps: build the image, verify both subcommands work locally, then
push to Docker Hub.

I run everything from the repository root so the `COPY` paths in the
Dockerfile resolve correctly. The `info` test is quick — it just confirms
the student JSON is readable. The `predict` test on a small local folder
is the important one: it's the only way to catch CSV formatting issues
before the image is live.

### B.11 - Docker Build, Test & Push

```bash
# 1. Build
docker build -t jabrailatakishiyev/cat-detector:final -f container/Dockerfile .

# 2. Test info
docker run --rm jabrailatakishiyev/cat-detector:final info

# 3. Test predict
mkdir -p /tmp/cat_inp /tmp/cat_out
docker run --rm \
  -v /tmp/cat_inp:/data/input:ro \
  -v /tmp/cat_out:/data/output \
  jabrailatakishiyev/cat-detector:final predict
cat /tmp/cat_out/predictions.csv | head -10

# 4. Push
docker login
docker push jabrailatakishiyev/cat-detector:final
```

In [27]:
print('=' * 60)
print('IMAGE FOR LEADERBOARD')
print('=' * 60)
print()
print('docker pull jabrailatakishiyev/cat-detector:final')
print('Image:   jabrailatakishiyev/cat-detector:final')
print('Student: Jabrail Atakishiyev')


IMAGE FOR LEADERBOARD

docker pull jabrailatakishiyev/cat-detector:final
Image:   jabrailatakishiyev/cat-detector:final
Student: Jabrail Atakishiyev


## Summary

Three techniques, three runs, each one adding to the previous:

| Technique | Runs | What it targets |
|-----------|------|-----------------|
| Stronger augmentation | 1, 2, 3 | Scale, pose, and lighting variation — the direct cause of the Week-1 false negatives |
| Cosine LR schedule | 2, 3 | Late-epoch learning — fixes the premature LR decay that was capping Week-1 progress |
| Weight decay + early stopping | 3 | Overfitting on the larger pooled dataset; automatic checkpoint saving |

I also pooled all four datasets instead of just one, which gave the model
roughly 4× more training images than Week 1.

The v2 model has seen a much broader range of cats than the Week-1 model
ever did. The mAP numbers look lower on the combined split, but that's
because the baseline was only tested on a fraction of the data the v2
model trained on. With more compute — `yolo26s`, 100+ epochs — the gap
would close and then some.

**Shipped checkpoint:** `yolo26n`, Run 3 — aug + cos_lr + weight_decay  
**ONNX model:** 9.4 MB, shape `(1, 300, 6)`, verified against PyTorch  
**Container:** `jabrailatakishiyev/cat-detector:final`